# Validate the Python minispect radiance-spectrum estimate

Load every example world-camera measurement, run both MATLAB `estimateRadianceSpectrumFromMinispect` and Python `estimate_radiance_spectrum_form_ms`, and quantify agreement for the reconstructed spectrum, wavelength sampling, residual norm (`fVal`), and per-channel fit errors. The notebook uses one MATLAB Engine session for the complete validation run.

In [ ]:
# Import the Python libraries used for paths, numerical calculations, tables, plotting,
# MATLAB-file loading, and communication with MATLAB.
from pathlib import Path
import sys
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.io
import matlab.engine


# First, define some paths we are going to reuse.
PROJECT_ROOT: Path = Path("/Users/zacharykelly/Documents/MATLAB/projects/lightLoggerAnalysis")
CODE_ROOT: Path = PROJECT_ROOT / "code"
EXAMPLE_DATA_DIR: Path = PROJECT_ROOT / "data" / "exampleWorldCameraImages"
SENSOR_UTILITY_DIR: Path = CODE_ROOT / "library" / "sensor_utility"

# Add our sensor utility folder to Python's module search path so ms_util can be imported.
sys.path.append(str(SENSOR_UTILITY_DIR))
import ms_util

print(f"Project root: {PROJECT_ROOT}")

In [ ]:
# Next, let's load in the measurement paths.
# glob finds every example MAT file whose name contains _AGCandMS_. Sorting the
# paths makes the processing and output order reproducible.
measurement_paths: list[Path] = sorted(EXAMPLE_DATA_DIR.glob("*_AGCandMS_*.mat"))
if not measurement_paths:
    raise FileNotFoundError(f"No example measurements found in {EXAMPLE_DATA_DIR}")

print(f"Found {len(measurement_paths)} measurements:")
for measurement_path in measurement_paths:
    print(f"  {measurement_path.name}")

In [ ]:
def compare_spectrum_results(
    measurement_name: str,
    matlab_spectrum: np.ndarray,
    python_spectrum: np.ndarray,
    matlab_sampling: np.ndarray,
    python_sampling: np.ndarray,
    matlab_f_val: float,
    python_f_val: float,
    matlab_fit_errors: np.ndarray,
    python_fit_errors: np.ndarray,
) -> tuple[dict[str, object], np.ndarray, np.ndarray]:
    """Calculate finite-value errors and compare non-finite masks."""

    # If the shape is not equal, this is a huge serious error.
    if matlab_spectrum.shape != python_spectrum.shape:
        raise ValueError(
            f"Shape mismatch for {measurement_name}: MATLAB {matlab_spectrum.shape}, "
            f"Python {python_spectrum.shape}."
        )

    # Each implementation must return one fit residual per sensor channel.
    if matlab_fit_errors.shape != python_fit_errors.shape:
        raise ValueError(
            f"Fit-error shape mismatch for {measurement_name}: MATLAB "
            f"{matlab_fit_errors.shape}, Python {python_fit_errors.shape}."
        )

    # The wavelength sampling descriptors must also have the same shape.
    if matlab_sampling.shape != python_sampling.shape:
        raise ValueError(
            f"Sampling shape mismatch for {measurement_name}: MATLAB "
            f"{matlab_sampling.shape}, Python {python_sampling.shape}."
        )

    # Find the wavelength samples that are finite in both spectra.
    finite_in_both: np.ndarray = np.isfinite(matlab_spectrum) & np.isfinite(python_spectrum)

    # Allocate arrays for the absolute and relative errors between the spectra.
    absolute_error: np.ndarray = np.full(matlab_spectrum.shape, np.nan, dtype=np.float64)
    relative_error: np.ndarray = np.full(matlab_spectrum.shape, np.nan, dtype=np.float64)

    # Find the absolute error at every wavelength that is finite in both results.
    absolute_error[finite_in_both] = np.abs(
        python_spectrum[finite_in_both] - matlab_spectrum[finite_in_both]
    )

    # Relative error requires division by the MATLAB result. Exclude zero-valued
    # MATLAB samples so that this calculation does not divide by zero.
    relative_denominator: np.ndarray = finite_in_both & (matlab_spectrum != 0)
    relative_error[relative_denominator] = (
        absolute_error[relative_denominator]
        / np.abs(matlab_spectrum[relative_denominator])
    )

    # Pull the valid errors into vectors so summary statistics can be calculated
    # without the NaNs used to mark invalid wavelengths in the plotted arrays.
    finite_absolute_error: np.ndarray = absolute_error[finite_in_both]
    finite_relative_error: np.ndarray = relative_error[np.isfinite(relative_error)]
    sampling_absolute_error: np.ndarray = np.abs(python_sampling - matlab_sampling)
    fit_errors_absolute_error: np.ndarray = np.abs(
        python_fit_errors - matlab_fit_errors
    )
    f_val_absolute_error: float = abs(python_f_val - matlab_f_val)
    f_val_relative_error: float = (
        f_val_absolute_error / abs(matlab_f_val)
        if matlab_f_val != 0
        else (0.0 if f_val_absolute_error == 0 else float("inf"))
    )

    # Record the same complementary comparisons used by the pipeline validation
    # notebook, along with checks for the returned wavelength sampling descriptor.
    metrics: dict[str, object] = {
        "measurement": measurement_name,
        "finite_wavelengths": int(np.count_nonzero(finite_in_both)),
        "max_absolute_error": float(np.max(finite_absolute_error)),
        "mean_absolute_error": float(np.mean(finite_absolute_error)),
        "rmse": float(np.sqrt(np.mean(finite_absolute_error ** 2))),
        "max_relative_error": float(np.max(finite_relative_error)) if finite_relative_error.size else 0.0,
        "inf_masks_equal": bool(np.array_equal(np.isinf(matlab_spectrum), np.isinf(python_spectrum))),
        "nan_masks_equal": bool(np.array_equal(np.isnan(matlab_spectrum), np.isnan(python_spectrum))),
        "allclose": bool(np.allclose(matlab_spectrum, python_spectrum, rtol=1e-12, atol=1e-12, equal_nan=True)),
        "sampling_max_absolute_error": float(np.max(sampling_absolute_error)),
        "sampling_allclose": bool(np.allclose(matlab_sampling, python_sampling, rtol=0, atol=0)),
        "matlab_f_val": matlab_f_val,
        "python_f_val": python_f_val,
        "f_val_absolute_error": f_val_absolute_error,
        "f_val_relative_error": f_val_relative_error,
        "f_val_allclose": bool(np.isclose(matlab_f_val, python_f_val, rtol=1e-12, atol=1e-12)),
        "fit_errors_max_absolute_error": float(np.max(fit_errors_absolute_error)),
        "fit_errors_allclose": bool(np.allclose(matlab_fit_errors, python_fit_errors, rtol=1e-12, atol=1e-12, equal_nan=True)),
    }
    return metrics, absolute_error, relative_error

In [ ]:
# Store one metrics row and all comparison outputs for every measurement.
comparison_rows: list[dict[str, object]] = []
comparison_outputs: dict[str, dict[str, object]] = {}

# Start MATLAB only once. tbUseProject activates lightLoggerAnalysis and adds
# the project plus its configured toolbox dependencies to the MATLAB path.
matlab_engine: object = matlab.engine.start_matlab()
matlab_engine.tbUseProject("lightLoggerAnalysis", nargout=0)

try:
    for measurement_path in measurement_paths:
        # Load the minispect structure in Python. simplify_cells converts the
        # MATLAB struct into a dictionary and removes singleton dimensions.
        python_data: dict[str, object] = scipy.io.loadmat(
            measurement_path,
            variable_names=["minispectValue"],
            simplify_cells=True,
        )
        minispect_value: dict[str, object] = python_data["minispectValue"]
        minispect_counts: np.ndarray = np.asarray(minispect_value["AS"])

        # MATLAB loads its native minispect struct directly from the same file.
        matlab_data: dict[str, object] = matlab_engine.load(
            str(measurement_path),
            "minispectValue",
            nargout=1,
        )

        # Run the MATLAB reference implementation using its native AS values.
        matlab_spectrum_result, matlab_sampling_result, matlab_f_val_result, matlab_fit_errors_result = (
            matlab_engine.estimateRadianceSpectrumFromMinispect(
                matlab_data["minispectValue"]["AS"],
                nargout=4,
            )
        )
        matlab_spectrum: np.ndarray = np.asarray(
            matlab_spectrum_result, dtype=np.float64
        ).reshape(-1)
        matlab_sampling: np.ndarray = np.asarray(
            matlab_sampling_result, dtype=np.float64
        ).reshape(-1)
        matlab_f_val: float = float(np.asarray(matlab_f_val_result).squeeze())
        matlab_fit_errors: np.ndarray = np.asarray(
            matlab_fit_errors_result, dtype=np.float64
        ).reshape(-1)

        # Run the Python reconstruction on the same ten AS7341 channel values.
        python_spectrum, python_sampling, python_f_val, python_fit_errors = (
            ms_util.estimate_radiance_spectrum_form_ms(minispect_counts)
        )

        # Calculate this measurement's errors, retain its values for visualization,
        # and add its summary metrics to the final table.
        metrics, absolute_error, relative_error = compare_spectrum_results(
            measurement_path.name,
            matlab_spectrum,
            python_spectrum,
            matlab_sampling,
            python_sampling,
            matlab_f_val,
            python_f_val,
            matlab_fit_errors,
            python_fit_errors,
        )
        comparison_rows.append(metrics)
        comparison_outputs[measurement_path.name] = {
            "minispect_counts": minispect_counts,
            "matlab_spectrum": matlab_spectrum,
            "python_spectrum": python_spectrum,
            "matlab_sampling": matlab_sampling,
            "python_sampling": python_sampling,
            "matlab_f_val": matlab_f_val,
            "python_f_val": python_f_val,
            "matlab_fit_errors": matlab_fit_errors,
            "python_fit_errors": python_fit_errors,
            "absolute_error": absolute_error,
            "relative_error": relative_error,
        }
# Always close MATLAB, even if loading or processing one measurement raises an error.
finally:
    matlab_engine.quit()

In [ ]:
# Combine the per-measurement dictionaries into one table and restore filename order.
comparison_table: pd.DataFrame = (
    pd.DataFrame(comparison_rows).sort_values("measurement").reset_index(drop=True)
)

# Display error values as ordinary decimals instead of scientific notation.
error_columns: list[str] = [
    "max_absolute_error",
    "mean_absolute_error",
    "rmse",
    "max_relative_error",
    "sampling_max_absolute_error",
    "matlab_f_val",
    "python_f_val",
    "f_val_absolute_error",
    "f_val_relative_error",
    "fit_errors_max_absolute_error",
]
display(comparison_table.style.format({column: "{:.20f}" for column in error_columns}))

In [ ]:
# Create one 1-by-3 comparison figure for every example measurement.
for measurement_path in measurement_paths:
    measurement_name: str = measurement_path.name
    output: dict[str, object] = comparison_outputs[measurement_name]

    minispect_counts: np.ndarray = output["minispect_counts"]
    matlab_spectrum: np.ndarray = output["matlab_spectrum"]
    python_spectrum: np.ndarray = output["python_spectrum"]
    matlab_sampling: np.ndarray = output["matlab_sampling"]
    absolute_error: np.ndarray = output["absolute_error"]
    matlab_f_val: float = output["matlab_f_val"]
    python_f_val: float = output["python_f_val"]

    # Expand MATLAB's [start, step, count] descriptor into wavelength values.
    wavelengths: np.ndarray = (
        matlab_sampling[0] + matlab_sampling[1] * np.arange(int(matlab_sampling[2]))
    )

    fig, axes = plt.subplots(1, 3, figsize=(18, 5), constrained_layout=True)
    fig.suptitle(measurement_name, fontsize=16, fontweight="bold")

    # Show the ten raw sensor values used by both implementations.
    channel_indices: np.ndarray = np.arange(1, minispect_counts.size + 1)
    axes[0].bar(channel_indices, minispect_counts)
    axes[0].set_title("Minispect AS values")
    axes[0].set_xlabel("Sensor channel")
    axes[0].set_ylabel("Raw count")

    # Overlay the MATLAB and Python reconstructions in the second panel so
    # their agreement can be inspected directly along the same wavelength axis.
    axes[1].plot(wavelengths, matlab_spectrum, color="tab:blue", linewidth=2, label="MATLAB")
    axes[1].plot(wavelengths, python_spectrum, color="tab:orange", linestyle="--", linewidth=2, label="Python")
    axes[1].set_title("Estimated radiance spectra")
    axes[1].set_xlabel("Wavelength (nm)")
    axes[1].set_ylabel("Spectral radiance")
    axes[1].set_xlim(400, 1000)
    axes[1].legend(loc="best")

    # Summarize the finite absolute errors so the final panel shows both the
    # wavelength-by-wavelength difference and its overall scale.
    finite_absolute_error: np.ndarray = absolute_error[np.isfinite(absolute_error)]
    mean_absolute_error: float = float(np.mean(finite_absolute_error))
    rmse: float = float(np.sqrt(np.mean(finite_absolute_error ** 2)))

    # Plot one consolidated error panel with the absolute error, its mean, and RMSE.
    axes[2].plot(wavelengths, absolute_error, color="tab:red", label="Absolute error")
    axes[2].axhline(mean_absolute_error, color="black", linestyle="--", label=f"Mean = {mean_absolute_error:.3e}")
    axes[2].axhline(rmse, color="tab:purple", linestyle=":", label=f"RMSE = {rmse:.3e}")
    axes[2].set_title("MATLAB vs. Python error")
    axes[2].set_xlabel("Wavelength (nm)")
    axes[2].set_ylabel("Absolute error")
    axes[2].set_xlim(400, 1000)
    axes[2].text(
        0.02,
        0.98,
        f"MATLAB fVal = {matlab_f_val:.6e}\nPython fVal = {python_f_val:.6e}",
        transform=axes[2].transAxes,
        verticalalignment="top",
    )
    axes[2].legend(loc="best")

    # Add a grid to every panel to make values and errors easier to inspect.
    for axis in axes:
        axis.grid(True, alpha=0.3)

    plt.show()